In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone

In [0]:
%run ./utils/functions

In [0]:
%run ./utils/validations

### Configure parameters

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

In [0]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
run_id = dbutils.widgets.get("run_id").strip()

current_state_table = f"{catalog}.{schema}.domains_classification_current"
history_table = f"{catalog}.{schema}.domains_classification_history"

### Detect changes

In [0]:
# Prefer the snapshot timestamp supplied by the orchestrator.
# Use one effective timestamp for all changes in this execution.
effective_ts = datetime.now(timezone.utc)

incoming = spark.table(current_state_table).select(
    "domain",
    "category",
    "feeds",
    "feed_count"
)

active = (
    spark.table(history_table)
    .filter(F.col("is_current"))
    .select(
        "domain",
        "category",
        "feeds",
        "feed_count"
    )
)

# compare to detect changes
comparison = (
    active.alias("old")
    .join(
        incoming.alias("new"),
        F.col("old.domain") == F.col("new.domain"),
        "full_outer",
    )
)

same_attributes = (
    F.col("old.category").eqNullSafe(F.col("new.category"))
    & F.sort_array(F.col("old.feeds")).eqNullSafe(
        F.col("new.feeds")
    )
)

# close disappeared domains and previous versions of changed domains
to_close = (
    comparison
    .filter(
        F.col("old.domain").isNotNull()
        & (
            F.col("new.domain").isNull()
            | ~same_attributes
        )
    )
    .select(
        F.col("old.domain").alias("domain"),
        F.col("old.category").alias("category"),
        F.col("old.feeds").alias("feeds"),
    )
    .withColumn("feed_count", F.size("feeds"))
    .withColumn("action", F.lit("close"))
)

# Insert new domains and new versions of changed domains.
to_insert = (
    comparison
    .filter(
        F.col("new.domain").isNotNull()
        & (
            F.col("old.domain").isNull()
            | ~same_attributes
        )
    )
    .select(
        F.col("new.domain").alias("domain"),
        F.col("new.category").alias("category"),
        F.col("new.feeds").alias("feeds"),
        F.col("new.feed_count").alias("feed_count"),
    )
    .withColumn("action", F.lit("insert"))
)

changes = (
    to_close.unionByName(to_insert)
    .withColumn("effective_ts", F.lit(effective_ts).cast("timestamp"))
    .withColumn("run_id", F.lit(run_id))
)

### Apply changes to history

In [0]:
# Close and insert versions atomically in one Delta transaction.
(
    DeltaTable.forName(spark, history_table)
    .alias("target")
    .merge(
        changes.alias("source"),
        """
        target.domain = source.domain
        AND target.is_current = true
        AND source.action = 'close'
        """,
    )
    .whenMatchedUpdate(
        set={
            "valid_to": "source.effective_ts",
            "is_current": "false",
            "closed_by_run_id": "source.run_id",
        }
    )
    .whenNotMatchedInsert(
        condition="source.action = 'insert'",
        values={
            "domain": "source.domain",
            "category": "source.category",
            "feeds": "source.feeds",
            "feed_count": "source.feed_count",
            "valid_from": "source.effective_ts",
            "valid_to": "CAST(NULL AS TIMESTAMP)",
            "is_current": "true",
            "run_id": "source.run_id",
            "closed_by_run_id": "CAST(NULL AS STRING)",
        },
    )
    .execute()
)